To do:
- Random sampling rather than time based
- DaCy segmentation
- Larger training set?
- No more effective batch sizing, actual batching at 256
- Slightly larger validation set (500?)
- 

- Hyperparameter tuning (Alpha, learning rate, batch size so on - not sure how to figure this out)
    - There is precedence for no hyperparameter tuning from the author of the OG NLI model that DEBATE is based on = Due to computational restrains and the points from this paper, no hyperparameter tuning was performed in this case. The model tuning in itself is also not the primary focus in this paper, but simply serves as a tool for the actual inquiry into blame in the Danish Parliament


In [1]:
%pip install -r "../requirements_bert.txt"

  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached tf_keras-2.20.1-py3-none-any.whl.metadata (1.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 59.6 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 37.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 93.3 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 53.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 112.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 82.7 MB/s  0:00:0

In [2]:
import torch
import transformers
import bitsandbytes
import accelerate
import datasets
import numpy as np
import pandas as pd
import keras
import json
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import Dataset
from keras.losses import binary_crossentropy
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, recall_score, classification_report, confusion_matrix

2026-02-25 10:32:38.103243: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
base_model_name = "jhu-clsp/mmBERT-base"

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
lora_config = LoraConfig(
    r=16,  # Low-rank dimension
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear"  # Fine-tuning all linear (classification, attention... layers)
)

lora_model = prepare_model_for_kbit_training(base_model)

# Enable gradient computation for LoRA parameters
for name, param in lora_model.named_parameters():
    if 'lora' in name.lower():
        param.requires_grad = True

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 3,416,096 || all params: 310,947,874 || trainable%: 1.0986


In [4]:
def tokenize_function(examples):
    return tokenizer(examples["text"], 
    padding="max_length", 
    truncation=True,
    max_length=512, # Padding to 512 to massively cut down on computation compared to base 8,192 tokens. 
    )

In [5]:
val_dataframe = pd.read_json("/work/RuneEgeskovTrust#9638//Bachelor/Bachelor_project/Model_data/validation_set.json")

val_dataframe = val_dataframe[['text', 'label']]

val_dataframe.rename(columns={'label': 'labels'}, inplace=True)

val_dataset = Dataset.from_pandas(val_dataframe)
tokenized_val = val_dataset.map(tokenize_function, batched=True, num_proc=16)


Map (num_proc=16): 100%|██████████| 258/258 [00:00<00:00, 467.69 examples/s]


In [6]:
dataframe_1_2_3_4_5 = pd.read_json("/work/RuneEgeskovTrust#9638/Bachelor/training_data/preproc_subset_1_2_3_4_5_cleaned_training_data.json")


In [16]:
dataframe_1_2_3_4_5 = dataframe_1_2_3_4_5[0:100]

dataframe_1_2_3_4_5 = dataframe_1_2_3_4_5[['text', 'label']]

dataframe_1_2_3_4_5.rename(columns={'label': 'labels'}, inplace=True)

In [11]:
data_for_merge = pd.read_json("/work/RuneEgeskovTrust#9638/Bachelor/training_data/preproc_data_for_tuning_final.json")

data_for_merge = data_for_merge[0:10000]

data_for_merge = data_for_merge[['text', 'label']]

data_for_merge.rename(columns={'label': 'labels'}, inplace=True)

In [17]:
frames = [data_for_merge, dataframe_1_2_3_4_5]

data = pd.concat(frames)

In [ ]:
#shuffling data 
data = data.sample(frac=1).reset_index(drop=True)

In [21]:
test_dataset = Dataset.from_pandas(data)

tokenized_test = test_dataset.map(tokenize_function, batched=True, num_proc=16)

Map (num_proc=16): 100%|██████████| 10100/10100 [00:00<00:00, 11745.39 examples/s]


In [ ]:
# 1. Smaller batch size with gradient accumulation
training_args = TrainingArguments(
    output_dir='/work/RuneEgeskovTrust#9638',
    #optim="paged_adamw_8bit",
    learning_rate=1e-4,
    num_train_epochs=3,
    per_device_train_batch_size=16,  # Reduced
    gradient_accumulation_steps=16,  # Effective batch = 256
    logging_steps=1,  # More frequent logging
    eval_strategy="epoch",  # Evaluate more frequently
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=8,
    remove_unused_columns=True,
    max_grad_norm=1.0,
    disable_tqdm=False,
    load_best_model_at_end=True,  # Add this
    metric_for_best_model="weighted BCE", # Add this
    greater_is_better = False
      
)

#OBS implement early stopping and more epochs, lower learning rate?

trainable params: 3,416,096 || all params: 310,947,874 || trainable%: 1.0986
Training samples: 3600
Validation samples: 258
Training steps per epoch: 14


In [10]:
def weighted_bincrossentropy(true, pred, weight_zero = 1, weight_one = 1):
    """
    Calculates weighted binary cross entropy. The weights are fixed to represent class imbalance in the dataset.
        
    For example if there are 10x as many positive classes as negative classes,
        if you adjust weight_zero = 1.0, weight_one = 0.1, then false positives 
        will be penalized 10 times as much as false negatives.

    """
  
    # calculate the binary cross entropy
    bin_crossentropy = binary_crossentropy(true, pred)
    
    # apply the weights
    weights = true * weight_one + (1. - true) * weight_zero
    #weights /= (weight_one + weight_zero) # Normalizing to be more consistent with regular BCE for comparison 
    weighted_bin_crossentropy = weights * bin_crossentropy 

    return np.mean(weighted_bin_crossentropy)

In [11]:

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    #From logits to probabilities
    probs_2d = np.exp(predictions) / np.exp(predictions).sum(axis=1, keepdims=True)
    probs = probs_2d[:, 1]  # positive class extraction
    
    weigthted_bce = weighted_bincrossentropy(labels, probs)
    keras_bce = binary_crossentropy(labels, probs)
    keras_bce = float(np.mean(keras_bce.numpy()))  # Converting from keras eagertensor to float value
    
    # Wrapping all metrics to floats for json serialization during model eval
    return {
        'keras_BCE': keras_bce,
        'weighted BCE': weigthted_bce,
        'recall': float(recall_score(labels, probs.round())),
        'precision': float(average_precision_score(labels, probs)),
        'accuracy': float(accuracy_score(labels, probs.round())), # Need rounding for these two computations (integer required)
        'f1': float(f1_score(labels, probs.round(), average='macro')), # macro f1 is better for imbalanced dataset
        'number_of_true_preds': sum(probs.round()),
        'number_of_true_labels': sum(labels)
    }

In [13]:
# Create custom trainer with weighted loss

from torch import nn
import torch

# Calculate class weights
label_counts = test_dataframe['labels'].value_counts()
total = len(test_dataframe)
weight_for_0 = total / (2 * label_counts[0])
weight_for_1 = total / (2 * label_counts[1])

print(f"Class 0 weight: {weight_for_0}")
print(f"Class 1 weight: {weight_for_1}")
print(f"Label distribution: {label_counts}")

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

Class 0 weight: 1.0
Class 1 weight: 1.0
Label distribution: labels
0    1800
1    1800
Name: count, dtype: int64


In [15]:
# Create the class weights tensor
class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float32)
print(class_weights)
# Use the custom trainer instead of the regular Trainer
trainer = WeightedLossTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_test,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    class_weights=class_weights  # Pass the weights here
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


tensor([1., 1.])


Epoch,Training Loss,Validation Loss,Keras Bce,Weighted bce,Recall,Precision,Accuracy,F1,Number Of True Preds,Number Of True Labels
1,0.587700,0.600283,0.600283,0.600283,0.863636,0.604226,0.701550,0.697731,141.000000,88
2,0.429100,0.498555,0.498555,0.498555,0.818182,0.724010,0.763566,0.753149,117.000000,88
3,0.277800,0.494904,0.494904,0.494904,0.840909,0.749380,0.767442,0.758367,120.000000,88


W0000 00:00:1761734876.026349   16692 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


TrainOutput(global_step=45, training_loss=8.07119000090493, metrics={'train_runtime': 92.3833, 'train_samples_per_second': 116.904, 'train_steps_per_second': 0.487, 'total_flos': 3793522603622400.0, 'train_loss': 8.07119000090493, 'epoch': 3.0})

In [ ]:
# Merge (now clean, no quantization artifacts)
model = trainer.model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("/work/MarkusLundsfrydJensen#1865/test_models_save/Option_2")
tokenizer.save_pretrained("/work/MarkusLundsfrydJensen#1865/test_models_save/Option_2")

('/work/MarkusLundsfrydJensen#1865/test_models_save/Option_2/tokenizer_config.json',
 '/work/MarkusLundsfrydJensen#1865/test_models_save/Option_2/special_tokens_map.json',
 '/work/MarkusLundsfrydJensen#1865/test_models_save/Option_2/tokenizer.json')

In [16]:
#OBS need to be made usable
class BlameDetectorDa_v0(object):

    def __init__(self, model, tokenizer, max_length):

        self.model = model
        self.max_length = max_length
        self.tokenizer = tokenizer
        #self.batch_size = batch_size

        self.model_initialization()

        return

    def model_initialization(self):
        #self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_path)
        self.model.eval()
            
        # Move to GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(self.device)
        self.model.to(self.device)

        print(f"Model loaded successfully on {self.device}")

        return

    def predict(self):
        """Make a prediction on a single text input."""
        # Tokenize input
        inputs = self.tokenizer(
            self.text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Move inputs to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Make prediction
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)
            predicted_class = torch.argmax(probabilities, dim=1).item()
            confidence = probabilities[0][predicted_class].item()
        
        return predicted_class, confidence, probabilities[0].cpu().numpy()

    def run_prediction(self, text):

        self.text = text
        predicted_class, confidence, probs = self.predict()
            
        return predicted_class, confidence


In [18]:
BD = BlameDetectorDa_v0(model = model_tester, tokenizer = tokenizer, max_length = 512)

cuda
Model loaded successfully on cuda


In [19]:
import json
json_path = "/work/MarkusLundsfrydJensen#1865/Bachelor_project/Model_data/validation_set.json"

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [21]:
for entry in data:
    text = entry["text"]

    entry["predicted_class"], entry["confidence"] = BD.run_prediction(text)

In [22]:
true_labels = 0
pred_true = 0
false_labels = 0
pred_false = 0

correct_pred = 0
incorrect_pred = 0

for entry in data:
    if entry["label"] == 1:
        true_labels += 1
    if entry["predicted_class"] == 1:
        pred_true +=1

    if entry["label"] == 0:
        false_labels += 1
    if entry["predicted_class"] == 0:
        pred_false +=1

    if entry["label"] == entry["predicted_class"]:
        correct_pred +=1

    if entry["label"] != entry["predicted_class"]:
        incorrect_pred +=1


print(correct_pred/len(data))

0.7558139534883721
